In [0]:
# ==========================================
# SILVER LAYER — Bloco 1: Setup
# ==========================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import re

CATALOG       = "workspace"
BRONZE_SCHEMA = "qap_2025_bronze"
SILVER_SCHEMA = "qap_2025_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
print(f"[OK] Schema {CATALOG}.{SILVER_SCHEMA} pronto.")

In [0]:
# ==========================================
# SILVER LAYER — Bloco 2: Contratos
# ==========================================

SILVER_CONFIG = [
    {
        "bronze": "production_monthly_csv_raw",
        "silver": "production_monthly",
        "pk":     ["month"],
        "required_cols": ["month", "production_target_units", "production_actual_units"],
        "casts": {
            "production_target_units":        "integer",
            "production_actual_units":        "integer",
            "absenteeism_target_events":      "integer",
            "absenteeism_actual_events":      "integer",
            "monthly_cost_target":            "double",
            "monthly_cost_actual":            "double",
            "production_loss_target_minutes": "integer",
            "production_loss_actual_minutes": "integer"
        }
    },
    {
        "bronze": "maintenance_daily_csv_raw",
        "silver": "maintenance_daily",
        "pk":     ["date"],
        "required_cols": ["date", "preventive_planned", "breakdown_events"],
        "casts": {
            "preventive_planned":  "integer",
            "preventive_done":     "integer",
            "preventive_not_done": "integer",
            "breakdown_events":    "integer",
            "mttr_minutes_avg":    "double"
        }
    },
    {
        "bronze": "engineering_daily_csv_raw",
        "silver": "engineering_daily",
        "pk":     ["date"],
        "required_cols": ["date", "avg_cycle_time_sec", "takt_time_sec"],
        "casts": {
            "avg_cycle_time_sec":              "double",
            "takt_time_sec":                   "double",
            "meets_demand":                    "boolean",
            "needs_improvement_to_meet_takt":  "boolean"
        }
    },
    {
        "bronze": "hr_employees_csv_raw",
        "silver": "hr_employees",
        "pk":     ["employee_id"],
        "required_cols": ["employee_id", "role", "team"],
        "casts": {
            "age":     "integer",
            "married": "boolean"
        }
    },
    {
        "bronze": "quality_daily_jsonl_raw",
        "silver": "quality_daily",
        "pk":     ["date", "team", "shift"],
        "required_cols": ["date", "team", "quality_actual_fpy"],
        "casts": {
            "quality_actual_fpy": "double",
            "quality_target_fpy": "double",
            "rework_happened":    "boolean",
            "rework_rate":        "double",
            "rework_units":       "integer"
        }
    },
    {
        "bronze": "safety_daily_json_raw",
        "silver": "safety_daily",
        "pk":     ["date"],
        "required_cols": ["date", "accident_occurred"],
        "casts": {
            "accident_occurred": "boolean",
            "near_miss_count":   "integer"
        },
        "is_json_array": True
    }
]

print(f"[OK] {len(SILVER_CONFIG)} contratos definidos.")

In [0]:
# ==========================================
# SILVER LAYER — Bloco 3: Funções
# ==========================================

def clean_column_names(df):
    """Padroniza nomes de colunas: remove espaços, caracteres especiais."""
    for col in df.columns:
        clean = re.sub(r"[^a-z0-9_]", "_", col.lower().strip())
        df = df.withColumnRenamed(col, clean)
    return df

def validate_data(df, required_cols):
    """Cria coluna _errors com lista de motivos de rejeição."""
    reasons = F.array().cast("array<string>")
    for col in required_cols:
        if col in df.columns:
            reasons = F.when(
                F.col(col).isNull(),
                F.array_union(reasons, F.array(F.lit(f"MISSING_{col}")))
            ).otherwise(reasons)
    return df.withColumn("_errors", reasons)

def upsert_to_silver(df, table_name, pk_cols):
    """MERGE: atualiza se existe, insere se é novo. Garante idempotência."""
    from delta.tables import DeltaTable
    full_name = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"

    if spark.catalog.tableExists(full_name):
        dt = DeltaTable.forName(spark, full_name)
        join_cond = " AND ".join([f"target.{c} = source.{c}" for c in pk_cols])
        dt.alias("target").merge(df.alias("source"), join_cond) \
          .whenMatchedUpdateAll() \
          .whenNotMatchedInsertAll() \
          .execute()
    else:
        df.write.format("delta").mode("overwrite") \
          .option("overwriteSchema", "true").saveAsTable(full_name)

print("[OK] Funções prontas.")

In [0]:
# ==========================================
# SILVER LAYER — Bloco 4: Execução
# ==========================================

for conf in SILVER_CONFIG:
    try:
        t_bronze = conf["bronze"]
        t_silver = conf["silver"]
        print(f"--- Processando: {t_silver} ---")

        # 1. LER DO BRONZE
        df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{t_bronze}")

        # 2. TRATAMENTO ESPECIAL: JSON Array (safety_daily)
        if conf.get("is_json_array"):
            schema_json = "array<struct<date:string,accident_occurred:boolean,near_miss_count:int>>"
            df = df.withColumn("parsed", F.from_json("value", schema_json)) \
                   .select(F.explode("parsed").alias("data"), "ingestion_ts") \
                   .select("data.*", "ingestion_ts")

        # 3. PADRONIZAR NOMES DE COLUNAS
        df = clean_column_names(df)

        # 4. CONVERTER TIPOS (Casts)
        for col_name, col_type in conf["casts"].items():
            if col_name in df.columns:
                df = df.withColumn(col_name, F.col(col_name).cast(col_type))

        # 5. VALIDAR QUALIDADE
        df = validate_data(df, conf["required_cols"])
        df = df.withColumn("_processing_ts", F.current_timestamp())

        # 6. SEPARAR VÁLIDOS E REJEITADOS
        df_bom  = df.filter(F.size("_errors") == 0).drop("_errors")
        df_ruim = df.filter(F.size("_errors") > 0)

        # 7. SALVAR VÁLIDOS (MERGE)
        count_bom = df_bom.count()
        if count_bom > 0:
            upsert_to_silver(df_bom, t_silver, conf["pk"])
            print(f"   [SUCESSO] {count_bom} linhas salvas em '{t_silver}'")
        else:
            print(f"   [AVISO] Nenhuma linha válida para '{t_silver}'")

        # 8. SALVAR REJEITADOS (Audit Trail)
        count_ruim = df_ruim.count()
        if count_ruim > 0:
            df_ruim.write.format("delta").mode("append") \
                   .option("overwriteSchema", "true") \
                   .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.rejected_{t_silver}")
            print(f"   [ALERTA] {count_ruim} linhas rejeitadas salvas em 'rejected_{t_silver}'")

    except Exception as e:
        print(f"   [FALHA] {conf['silver']}: {str(e)}")

print("\n[FIM] Pipeline Silver concluído.")

In [0]:
# ==========================================
# SILVER LAYER — Bloco 5: Bridge Enriquecida
# ==========================================
print("=== Criando Bridge enriquecida ===")

# 1. BASE: Calendário de turnos (fonte de verdade de equipe + data + turno)
df_schedule = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.shift_schedule_2025_raw")

df_day = df_schedule.select(
    F.col("date").cast("date").alias("date"),
    F.col("day_team").alias("team"),
    F.lit("DAY").alias("shift"),
    F.col("day_start").alias("shift_start"),
    F.col("day_end").alias("shift_end")
)
df_night = df_schedule.select(
    F.col("date").cast("date").alias("date"),
    F.col("night_team").alias("team"),
    F.lit("NIGHT").alias("shift"),
    F.col("night_start").alias("shift_start"),
    F.col("night_end").alias("shift_end")
)

df_bridge = df_day.union(df_night) \
    .withColumn("bridge_key", F.concat_ws("_", F.col("date").cast("string"), F.col("team"), F.col("shift")))

# 2. JOIN: Qualidade (fpy, rework)
df_quality = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.quality_daily") \
    .select("date", "team", "shift",
            "quality_actual_fpy", "quality_target_fpy",
            "rework_happened", "rework_rate", "rework_units") \
    .withColumn("date", F.col("date").cast("date"))

df_bridge = df_bridge.join(df_quality, on=["date", "team", "shift"], how="left")

# 3. JOIN: Manutenção (breakdowns, mttr)
df_maint = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.maintenance_daily") \
    .select("date", "breakdown_events", "mttr_minutes_avg",
            "preventive_planned", "preventive_done") \
    .withColumn("date", F.col("date").cast("date"))

df_bridge = df_bridge.join(df_maint, on="date", how="left")

# 4. JOIN: Engenharia (cycle time vs takt)
df_eng = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.engineering_daily") \
    .select("date", "avg_cycle_time_sec", "takt_time_sec",
            "meets_demand", "needs_improvement_to_meet_takt") \
    .withColumn("date", F.col("date").cast("date"))

df_bridge = df_bridge.join(df_eng, on="date", how="left")

# 5. KPIs CALCULADOS NA SILVER (prontos para a Gold consumir)
df_bridge = df_bridge \
    .withColumn("cycle_gap_pct",
        F.round(
            (F.col("avg_cycle_time_sec") - F.col("takt_time_sec")) / F.col("takt_time_sec"), 4
        )) \
    .withColumn("fpy_gap",
        F.round(F.col("quality_actual_fpy") - F.col("quality_target_fpy"), 4)) \
    .withColumn("quality_status",
        F.when(F.col("quality_actual_fpy") >= 0.95, "GOOD")
         .when(F.col("quality_actual_fpy") >= 0.85, "WARNING")
         .when(F.col("quality_actual_fpy").isNull(), "NO_DATA")
         .otherwise("CRITICAL")) \
    .withColumn("silver_ts", F.current_timestamp()) \
    .withColumn("_processing_ts", F.current_timestamp())

# 6. SALVAR
df_bridge.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.bridge_team_day_context")

print(f"   [SUCESSO] bridge_team_day_context: {df_bridge.count()} linhas")
print(f"   Colunas: {df_bridge.columns}")

In [0]:
# ==========================================
# SILVER LAYER — Bloco 6: Resumo
# ==========================================

tabelas = spark.sql(f"SHOW TABLES IN {CATALOG}.{SILVER_SCHEMA}") \
               .filter("tableName NOT LIKE 'rejected_%'")

print("=" * 50)
print("SILVER LAYER — TABELAS CRIADAS")
print("=" * 50)
for row in tabelas.collect():
    nome = f"{CATALOG}.{SILVER_SCHEMA}.{row['tableName']}"
    try:
        qtd = spark.table(nome).count()
        print(f"  {row['tableName']:<35} {qtd:>6} linhas")
    except:
        print(f"  {row['tableName']:<35} [erro ao contar]")
print("=" * 50)
print("Pronto para rodar a Gold!")